# 04 - Evaluation

This notebook has two parts:

1. **Classifier evaluation** — runs each trained model (`.pth`) over the real
   test set and reports accuracy, per-class precision/recall/F1, and macro/
   weighted averages. Uses `evaluate.py`.
2. **Quality scoring and grading** — exercises the rule-based grading layer
   in `grading.py` with dummy `{class, confidence}` inputs.

---

## Part 1 — Classifier evaluation

In [3]:
import sys
from pathlib import Path

import torch

# Make the repo root importable
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from task2_3_4_cv_quality.src.evaluate import (
    compute_metrics,
    confusion_matrix_df,
    evaluate_model,
    print_results_table,
)
from task2_3_4_cv_quality.src.model import get_model
from task2_3_4_cv_quality.src.train import create_dataloaders

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [4]:
bundle = create_dataloaders(verbose=False)
class_names = bundle.classes
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names[:3]} ... {class_names[-3:]}")
print(f"Test samples: {len(bundle.test_loader.dataset)}")

Classes (28): ['Apple__Healthy', 'Apple__Rotten', 'Banana__Healthy'] ... ['Strawberry__Rotten', 'Tomato__Healthy', 'Tomato__Rotten']
Test samples: 29291


In [5]:
MODELS_DIR = REPO_ROOT / "task2_3_4_cv_quality" / "models"

checkpoints = [
    ("baseline_cnn_best.pth",    "custom_cnn"),
    ("resnet50_best.pth",        "resnet50"),
    ("efficientnet_b0_best.pth", "efficientnet_b0"),
]

all_results = {}

for filename, model_type in checkpoints:
    path = MODELS_DIR / filename
    print(f"\nLoading {filename} ({model_type})...")

    model = get_model(num_classes=num_classes, model_type=model_type, pretrained=False)
    checkpoint = torch.load(path, map_location=device)
    # Mohamed saved full training checkpoints, so extract the weights
    state = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state)

    eval_result = evaluate_model(model, bundle.test_loader, class_names, device)
    metrics = compute_metrics(eval_result.y_true, eval_result.y_pred, class_names)

    print_results_table(metrics, title=f"{model_type} ({filename})")
    all_results[model_type] = {"eval_result": eval_result, "metrics": metrics}


Loading baseline_cnn_best.pth (custom_cnn)...


C:\Users\Konstantinos\AppData\Roaming\Python\Python312\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


custom_cnn (baseline_cnn_best.pth)
Samples evaluated : 29291
Overall accuracy  : 0.9563
--------------------------------------------------------------------------------
               class precision recall     f1  support
      Apple__Healthy    0.9706 0.9348 0.9524     2438
       Apple__Rotten    0.9486 0.9188 0.9334     2930
     Banana__Healthy    0.9586 0.9850 0.9716     2000
      Banana__Rotten    0.9907 0.9871 0.9889     2800
 Bellpepper__Healthy    0.9629 0.9771 0.9699      611
  Bellpepper__Rotten    0.8289 0.9019 0.8639      591
     Carrot__Healthy    0.9702 0.9452 0.9575      620
      Carrot__Rotten    0.8805 0.9397 0.9091      580
   Cucumber__Healthy    0.9380 0.9951 0.9657      608
    Cucumber__Rotten    0.9628 0.9595 0.9611      593
      Grape__Healthy    0.9756 1.0000 0.9877      200
       Grape__Rotten    0.9851 0.9900 0.9875      200
      Guava__Healthy    0.9755 0.9950 0.9851      200
       Guava__Rotten    0.8609 0.9900 0.9209      200
     Jujube__Healthy 

C:\Users\Konstantinos\AppData\Roaming\Python\Python312\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


resnet50 (resnet50_best.pth)
Samples evaluated : 29291
Overall accuracy  : 0.9968
--------------------------------------------------------------------------------
               class precision recall     f1  support
      Apple__Healthy    1.0000 1.0000 1.0000     2438
       Apple__Rotten    0.9993 0.9942 0.9967     2930
     Banana__Healthy    1.0000 0.9995 0.9997     2000
      Banana__Rotten    1.0000 0.9996 0.9998     2800
 Bellpepper__Healthy    0.9984 0.9951 0.9967      611
  Bellpepper__Rotten    0.9832 0.9898 0.9865      591
     Carrot__Healthy    0.9935 0.9806 0.9870      620
      Carrot__Rotten    0.9746 0.9914 0.9829      580
   Cucumber__Healthy    0.9935 1.0000 0.9967      608
    Cucumber__Rotten    0.9966 0.9848 0.9907      593
      Grape__Healthy    0.9950 1.0000 0.9975      200
       Grape__Rotten    1.0000 0.9950 0.9975      200
      Guava__Healthy    1.0000 1.0000 1.0000      200
       Guava__Rotten    1.0000 1.0000 1.0000      200
     Jujube__Healthy    1.0

C:\Users\Konstantinos\AppData\Roaming\Python\Python312\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


efficientnet_b0 (efficientnet_b0_best.pth)
Samples evaluated : 29291
Overall accuracy  : 0.9961
--------------------------------------------------------------------------------
               class precision recall     f1  support
      Apple__Healthy    0.9971 0.9967 0.9969     2438
       Apple__Rotten    0.9997 0.9911 0.9954     2930
     Banana__Healthy    0.9990 0.9995 0.9993     2000
      Banana__Rotten    0.9993 1.0000 0.9996     2800
 Bellpepper__Healthy    0.9935 0.9984 0.9959      611
  Bellpepper__Rotten    0.9732 0.9848 0.9790      591
     Carrot__Healthy    0.9903 0.9855 0.9879      620
      Carrot__Rotten    0.9811 0.9862 0.9837      580
   Cucumber__Healthy    0.9951 1.0000 0.9975      608
    Cucumber__Rotten    0.9983 0.9798 0.9889      593
      Grape__Healthy    0.9950 1.0000 0.9975      200
       Grape__Rotten    1.0000 0.9950 0.9975      200
      Guava__Healthy    1.0000 1.0000 1.0000      200
       Guava__Rotten    0.9901 1.0000 0.9950      200
     Jujube__

In [6]:
import pandas as pd

summary = pd.DataFrame([
    {
        "model": name,
        "accuracy": r["metrics"].accuracy,
        "macro_f1": r["metrics"].macro["f1"],
        "weighted_f1": r["metrics"].weighted["f1"],
    }
    for name, r in all_results.items()
])
summary = summary.sort_values("weighted_f1", ascending=False).reset_index(drop=True)
summary

,model,accuracy,macro_f1,weighted_f1
0,resnet50,0.996791,0.995523,0.996796
1,efficientnet_b0,0.996142,0.994565,0.996145
2,custom_cnn,0.956301,0.951323,0.956539


In [7]:
best_model_name = summary.iloc[0]["model"]
best_result = all_results[best_model_name]["eval_result"]

print(f"Confusion matrix (row-normalised) for {best_model_name}:")
confusion_matrix_df(best_result, normalize="true").round(2)

Confusion matrix (row-normalised) for resnet50:


pred,Apple__Healthy,Apple__Rotten,Banana__Healthy,Banana__Rotten,Bellpepper__Healthy,Bellpepper__Rotten,Carrot__Healthy,Carrot__Rotten,Cucumber__Healthy,Cucumber__Rotten,...,Orange__Healthy,Orange__Rotten,Pomegranate__Healthy,Pomegranate__Rotten,Potato__Healthy,Potato__Rotten,Strawberry__Healthy,Strawberry__Rotten,Tomato__Healthy,Tomato__Rotten
true,,,,,,,,,,,,,,,,,,,,,
Apple__Healthy,1.0,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Apple__Rotten,0.0,0.99,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Banana__Healthy,0.0,0.00,1.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Banana__Rotten,0.0,0.00,0.0,1.0,0.0,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Bellpepper__Healthy,0.0,0.00,0.0,0.0,1.0,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Bellpepper__Rotten,0.0,0.00,0.0,0.0,0.0,0.99,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.01,0.0,0.0,0.0,0.00
Carrot__Healthy,0.0,0.00,0.0,0.0,0.0,0.00,0.98,0.02,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Carrot__Rotten,0.0,0.00,0.0,0.0,0.0,0.00,0.01,0.99,0.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00
Cucumber__Healthy,0.0,0.00,0.0,0.0,0.0,0.00,0.00,0.00,1.00,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00


---

## Part 2 — Quality scoring and grading

[Mohamed's existing grading notebook content continues from here...]



This notebook exercises the rule-based grading layer defined in
`task2_3_4_cv_quality/src/grading.py`.

## Heuristic overview

The grading layer runs **after** the classifier. It consumes a clean,
post-processed prediction dictionary of the form:

```python
{"predicted_class": "freshapples", "confidence": 0.91}
```

and returns three proxy quality percentages on a 0-100 scale plus a
letter grade and a recommendation. The heuristic is:

- **`fresh*`** classes
  - `color_score` and `ripeness_score` track confidence directly (`conf * 100`).
  - `size_score` is moderately high and less sensitive (`70 + conf * 30`).
- **`rotten*`** classes
  - higher confidence means the model is more sure the item is rotten, so
    quality scores **decrease** as confidence rises.
  - `color_score` and `ripeness_score` are harshly reduced; `size_score`
    is reduced more gently.

Grade = `A` if `min(scores) >= 80`, `B` if `>= 60`, otherwise `C`. Taking
the minimum means a single weak dimension pulls the grade down.

> **Note.** These are *proxy* quality scores derived from the predicted
> class and classifier confidence. They are **not** direct sensor
> measurements of real colour, size or ripeness.

In [8]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from task2_3_4_cv_quality.src.grading import (
    assign_grade,
    compute_quality_scores,
    get_recommendation,
    grade_produce,
)

## Dummy examples

Four hand-picked cases cover both freshness states at high and medium
confidence levels.

In [9]:
examples = [
    {"predicted_class": "Apple__Healthy",   "confidence": 0.98},
    {"predicted_class": "Banana__Rotten",   "confidence": 0.92},
    {"predicted_class": "Tomato__Rotten",   "confidence": 0.55},
]

results = [grade_produce(ex) for ex in examples]
for r in results:
    print(r)

{'predicted_class': 'Apple__Healthy', 'produce_type': 'Apple', 'state': 'Healthy', 'confidence': 0.98, 'color_score': 98, 'size_score': 99, 'ripeness_score': 98, 'grade': 'A', 'recommendation': 'Premium quality - sell at full price in premium display'}
{'predicted_class': 'Banana__Rotten', 'produce_type': 'Banana', 'state': 'Rotten', 'confidence': 0.92, 'color_score': 3, 'size_score': 52, 'ripeness_score': 3, 'grade': 'C', 'recommendation': 'Low quality - heavy discount or remove from sale'}
{'predicted_class': 'Tomato__Rotten', 'produce_type': 'Tomato', 'state': 'Rotten', 'confidence': 0.55, 'color_score': 18, 'size_score': 59, 'ripeness_score': 16, 'grade': 'C', 'recommendation': 'Low quality - heavy discount or remove from sale'}


## Tabular view

Same results rendered as a compact table for easy comparison.

In [10]:
header = ["predicted_class", "confidence", "color", "size", "ripeness", "grade", "recommendation"]
widths = [18, 10, 5, 5, 8, 5, 60]

def _fmt_row(values):
    return "  ".join(f"{str(v):<{w}}" for v, w in zip(values, widths))

print(_fmt_row(header))
print("-" * (sum(widths) + 2 * (len(widths) - 1)))
for r in results:
    print(_fmt_row([
        r["predicted_class"],
        f"{r['confidence']:.2f}",
        r["color_score"],
        r["size_score"],
        r["ripeness_score"],
        r["grade"],
        r["recommendation"],
    ]))

predicted_class     confidence  color  size   ripeness  grade  recommendation                                              
---------------------------------------------------------------------------------------------------------------------------
Apple__Healthy      0.98        98     99     98        A      Premium quality - sell at full price in premium display     
Banana__Rotten      0.92        3      52     3         C      Low quality - heavy discount or remove from sale            
Tomato__Rotten      0.55        18     59     16        C      Low quality - heavy discount or remove from sale            


## Helper functions in isolation

The intermediate functions can also be called directly.

In [11]:
sample = {"predicted_class": "Orange__Healthy", "confidence": 0.84}
scores = compute_quality_scores(sample)
print("scores        :", scores)
grade = assign_grade(scores)
print("grade         :", grade)
print("recommendation:", get_recommendation(grade))

scores        : {'predicted_class': 'Orange__Healthy', 'produce_type': 'Orange', 'state': 'Healthy', 'confidence': 0.84, 'color_score': 84, 'size_score': 95, 'ripeness_score': 84}
grade         : A
recommendation: Premium quality - sell at full price in premium display


## Input validation

Malformed inputs raise clear exceptions so integration bugs surface
early.

In [12]:
bad_inputs = [
    {"predicted_class": "Apple__Healthy"},
    
    {"predicted_class": "Apple__Healthy", "confidence": 1.7},
    

    {"predicted_class": "", "confidence": 0.5},
    {"predicted_class": "mystery_item", "confidence": 0.9},
    
    {"predicted_class": "freshapples", "confidence": 0.8},
    
    {"predicted_class": "Apple__Moldy", "confidence": 0.7},
    
    "not a dict",
]

print("Testing error handling:\n")
for i, bad in enumerate(bad_inputs, 1):
    try:
        grade_produce(bad)
        print(f"Test {i}: ❌ Should have raised an error!")
    except Exception as exc:
        print(f"Test {i}: {type(exc).__name__:<15} → {exc}")

Testing error handling:

Test 1: KeyError        → "model_output missing 'confidence'"
Test 2: ValueError      → Confidence must be in [0, 1], got 1.7
Test 3: ValueError      → Invalid class name format: ''. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 4: ValueError      → Invalid class name format: 'mystery_item'. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 5: ValueError      → Invalid class name format: 'freshapples'. Expected format: 'ProduceType__State' (e.g., 'Apple__Healthy')
Test 6: ValueError      → Invalid state: 'Moldy'. Expected 'Healthy' or 'Rotten'.
Test 7: KeyError        → "model_output missing 'predicted_class'"


## Summary

- Scores are derived **only** from the predicted class + confidence; no
  image-level measurements are made here.
- Grade A/B/C comes from the minimum of the three scores, so a single
  weak dimension can downgrade an otherwise strong item.
- Recommendations are keyed off the grade and can be overridden by
  editing `RECOMMENDATIONS` in `grading.py`.